In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


In [2]:
df_law = pd.read_csv("bar_pass_prediction.csv")
columns_to_keep = ["age", "decile1", "decile3", "fam_inc", "lsat", "ugpa", "gender", "race1", "fulltime", "bar"]
df_law = df_law[columns_to_keep]

#count of missing values in each column
missing_values = df_law.isnull().sum()

df_law.dropna(inplace=True)
# drop age since they don't make sense 
df_law.drop(columns=['age'], inplace=True)

In [3]:
# Encode categorical variables
df_law['gender'] = pd.factorize(df_law['gender'])[0] # 0 for female, 1 for male
df_law['race1'] = pd.factorize(df_law['race1'])[0] # 0 for white, 1 for hispanic, 2 for asian, 3 for black, 4 for other
df_law['bar'] = pd.factorize(df_law['bar'])[0] # 0 for passed first time, 1 for failed, 2 for passed second time

# Reset index after dropping rows
df_law.reset_index(drop=True, inplace=True)

# change feture names for consitency 
df_law.rename(columns={'race1': 'race', 'bar': 'target_law'}, inplace=True)

In [4]:
# imagine we only take 0 as passed first try and all the rest (so second try also) as failed first try 
df_law['target_law'] = df_law['target_law'].replace({2: 1})
# this slightly changes the target distibution favouring the ML model performance

In [5]:
df_law.shape

(20427, 9)

In [6]:
# define protected attributes 
protected_attributes = ["gender", "race"]

In [7]:
df_law

,decile1,decile3,fam_inc,lsat,ugpa,gender,race,fulltime,target_law
0,10.0,10.0,5.0,44.0,3.5,0,0,1.0,0
1,5.0,4.0,4.0,29.0,3.5,0,0,1.0,0
2,3.0,2.0,1.0,36.0,3.5,1,0,1.0,0
3,7.0,4.0,4.0,39.0,3.5,1,0,1.0,0
4,9.0,8.0,4.0,48.0,3.5,1,0,1.0,0
...,...,...,...,...,...,...,...,...,...
20422,3.0,1.0,2.0,26.5,1.8,1,3,1.0,1
20423,3.0,1.0,3.0,19.7,1.8,1,3,1.0,1
20424,7.0,8.0,3.0,36.0,1.8,1,3,2.0,0
20425,10.0,10.0,3.0,44.0,1.5,1,0,2.0,0


In [8]:
# feature categorization including all features
numerical_features = ["lsat", "ugpa"]
ordinal_features = ["decile1", "decile3", "fam_inc"]
nominal_features = ["gender", "race", "fulltime"]

# Define categorical features (ordinal + nominal)
categorical_features = ordinal_features + nominal_features

feature_desc = ['the decile to which the student belongs in the school given his grades in Year 1.', 
                'the decile to which the student belongs in the school given his grades in Year 3.',
                'the family income bracket of the student.',
                'the LSAT score of the student.',
                'the undergraduate GPA of the student.',
                'the gender of the student.',
                'the racial background of the student.',
                'whether the student will work full-time or part-time.'
]

In [9]:
#Load Data and Split
random_state = 1234
law_train, law_test = train_test_split(df_law, test_size=0.2, random_state=random_state)

law_train.to_parquet("train_cleaned.parquet")
law_test.to_parquet("test_cleaned.parquet")

# Separate features and target
x_train = law_train.drop("target_law", axis=1)
y_train = law_train["target_law"]
x_test = law_test.drop("target_law", axis=1)
y_test = law_test["target_law"] 

In [10]:
# random forest model

# Create training and test sets without protected attributes for the model
x_train_model = x_train.drop(columns=protected_attributes)
x_test_model = x_test.drop(columns=protected_attributes)

rf_model=RandomForestClassifier(random_state=random_state, class_weight='balanced')
rf_model.fit(x_train_model, y_train)
predictions=rf_model.predict(x_test_model)
target_pred_proba = rf_model.predict_proba(x_test_model)[:, 1]  # Get probability for class 1
# evaluate model based on accuracy and AUC metrics
accuracy = accuracy_score(y_test, predictions)
auc = roc_auc_score(y_test, target_pred_proba)
print("Prediction Accuracy:", accuracy)
print("AUC:", auc)

# Save the model
with open('RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Prediction Accuracy: 0.872491434165443
AUC: 0.8179518450845695


In [11]:
x_train_model.head()

,decile1,decile3,fam_inc,lsat,ugpa,fulltime
15735,6.0,5.0,2.0,38.0,3.1,1.0
782,10.0,10.0,4.0,39.0,3.5,1.0
10016,5.0,5.0,1.0,30.0,2.9,1.0
3507,10.0,10.0,4.0,40.0,3.4,1.0
19351,7.0,9.0,5.0,40.5,3.8,1.0


In [12]:
# Define value mappings 
FEATURE_MAPPINGS = {
    'gender': {0: 'Female', 1: 'Male'},
    'race1': {0: 'White', 1: 'Black', 2: 'Hispanic', 3: 'Asian', 4: 'Native American'},
    'fulltime': {1: 'Full-time', 1.0: 'Full-time', 2: 'Part-time', 2.0: 'Part-time'},
    'fam_inc': {1: 'Low', 2: 'Lower-middle', 3: 'Middle', 4: 'Upper-middle', 5: 'High'}
}

# Helper function to map categorical values to readable names
def map_value(feature_name, value):
    """Map encoded values to readable names"""
    if feature_name in FEATURE_MAPPINGS:
        mapping = FEATURE_MAPPINGS[feature_name]
        if value in mapping:
            return mapping[value]
        if float(value) in mapping:
            return mapping[float(value)]
    return str(value)

# Build feature dataframe with 
feature_data = []
for i, col in enumerate(x_train.columns):
    row = {
        "feature_name": col,
        "feature_desc": feature_desc[i]
    }
    feature_data.append(row)

feature_desc_df = pd.DataFrame(feature_data)     

dataset_description="The dataset contains information about students who took the bar exam, including their academic performance, family background, and demographic information. Each row is a student."
target_description="The target variable is whether the student passed the bar exam on the first try (0 = passed, 1 = failed)"
task_description="Predict whether a student will pass the bar exam on the first try given all the other features"

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [13]:
dataset_info 

{'dataset_description': 'The dataset contains information about students who took the bar exam, including their academic performance, family background, and demographic information. Each row is a student.',
 'target_description': 'The target variable is whether the student passed the bar exam on the first try (0 = passed, 1 = failed)',
 'task_description': 'Predict whether a student will pass the bar exam on the first try given all the other features',
 'feature_description':   feature_name                                       feature_desc
 0      decile1  the decile to which the student belongs in the...
 1      decile3  the decile to which the student belongs in the...
 2      fam_inc          the family income bracket of the student.
 3         lsat                     the LSAT score of the student.
 4         ugpa              the undergraduate GPA of the student.
 5       gender                         the gender of the student.
 6         race              the racial background 